Phase 5 — Feature Engineering

This notebook engineers time-based features (Task 14) and reviews geographic features (Task 15) from the validated Master Dataset, then finalizes the candidate Collision + Vehicle feature inventory (Task 16). Raw date/time fields are inspected before any derivation, per the project's detect-before-transform principle.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

data_path = Path("C:/Users/ACER/Downloads/New folder/cleaned_datasets")

master_df = pd.read_csv(data_path / "master_dataset_v1.csv", dtype={"collision_index": str})

print("Master Dataset shape:", master_df.shape)
master_df[["collision_index", "date", "time", "day_of_week"]].head()

Master Dataset shape: (513801, 61)


C:\Users\ACER\AppData\Local\Temp\ipykernel_18648\977430837.py:7: DtypeWarning: Columns (2: collision_ref_no) have mixed types. Specify dtype option on import or set low_memory=False.
  master_df = pd.read_csv(data_path / "master_dataset_v1.csv", dtype={"collision_index": str})


,collision_index,date,time,day_of_week
0,202417H105824,03/10/2024,16:06,5
1,202517M115025,29/11/2025,16:34,7
2,202517L306225,01/06/2025,22:07,1
3,202417S104724,20/04/2024,14:40,7
4,202517S106525,16/05/2025,07:25,6


Task 14.1 — Inspect Raw Date/Time Fields

Before deriving any time-based features, check the actual format and range of date and time, and confirm day_of_week is already a clean categorical field (per the STATS19 guide: 1=Sunday...7=Saturday) rather than something needing re-derivation.

In [2]:
print("date dtype:", master_df["date"].dtype)
print("date sample values:", master_df["date"].head(3).tolist())
print("date range:", master_df["date"].min(), "to", master_df["date"].max())

print("\ntime dtype:", master_df["time"].dtype)
print("time sample values:", master_df["time"].head(3).tolist())

print("\nday_of_week distribution:")
print(master_df["day_of_week"].value_counts().sort_index())

print("\nAny missing date/time/day_of_week?")
print("date:", master_df["date"].isna().sum())
print("time:", master_df["time"].isna().sum())
print("day_of_week:", master_df["day_of_week"].isna().sum())

date dtype: str
date sample values: ['03/10/2024', '29/11/2025', '01/06/2025']
date range: 01/01/2021 to 31/12/2025

time dtype: str
time sample values: ['16:06', '16:34', '22:07']

day_of_week distribution:
day_of_week
1    57781
2    70677
3    74829
4    76671
5    77954
6    85311
7    70578
Name: count, dtype: int64

Any missing date/time/day_of_week?
date: 0
time: 0
day_of_week: 0


Task 14.2 — Parse Date/Time and Derive Time Features

Parse date (day-first format) and time into proper datetime/hour values, then derive month, season (documented Northern Hemisphere mapping: Dec-Feb=Winter, Mar-May=Spring, Jun-Aug=Summer, Sep-Nov=Autumn), is_weekend (from day_of_week, where 1=Sunday and 7=Saturday per the STATS19 guide), hour_of_day, and time_of_day_bucket with explicitly documented boundaries: Night (00-05), Morning Rush (06-09), Midday (10-15), Evening Rush (16-18), Evening (19-23).

In [3]:
# Parse date (day-first) and time
master_df["date_parsed"] = pd.to_datetime(master_df["date"], dayfirst=True, format="%d/%m/%Y")
master_df["hour_of_day"] = pd.to_datetime(master_df["time"], format="%H:%M").dt.hour

# Confirm true chronological range now that it's parsed
print("Parsed date range:", master_df["date_parsed"].min(), "to", master_df["date_parsed"].max())

# month
master_df["month"] = master_df["date_parsed"].dt.month

# season (documented Northern Hemisphere mapping)
def month_to_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"

master_df["season"] = master_df["month"].apply(month_to_season)

# is_weekend (1=Sunday, 7=Saturday per STATS19 guide)
master_df["is_weekend"] = master_df["day_of_week"].isin([1, 7]).astype(int)

# time_of_day_bucket (documented boundaries)
def hour_to_bucket(hour):
    if 0 <= hour <= 5:
        return "Night"
    elif 6 <= hour <= 9:
        return "Morning Rush"
    elif 10 <= hour <= 15:
        return "Midday"
    elif 16 <= hour <= 18:
        return "Evening Rush"
    else:
        return "Evening"

master_df["time_of_day_bucket"] = master_df["hour_of_day"].apply(hour_to_bucket)

# Validate
print("\nhour_of_day range:", master_df["hour_of_day"].min(), "to", master_df["hour_of_day"].max())
print("\nmonth distribution:")
print(master_df["month"].value_counts().sort_index())
print("\nseason distribution:")
print(master_df["season"].value_counts())
print("\nis_weekend distribution:")
print(master_df["is_weekend"].value_counts())
print("\ntime_of_day_bucket distribution:")
print(master_df["time_of_day_bucket"].value_counts())

Parsed date range: 2021-01-01 00:00:00 to 2025-12-31 00:00:00

hour_of_day range: 0 to 23

month distribution:
month
1     39815
2     35179
3     40094
4     40107
5     45204
6     46511
7     45204
8     42591
9     45083
10    45870
11    46835
12    41308
Name: count, dtype: int64

season distribution:
season
Autumn    137788
Summer    134306
Spring    125405
Winter    116302
Name: count, dtype: int64

is_weekend distribution:
is_weekend
0    385442
1    128359
Name: count, dtype: int64

time_of_day_bucket distribution:
time_of_day_bucket
Midday          184238
Evening Rush    123623
Evening          91195
Morning Rush     85559
Night            29186
Name: count, dtype: int64


In [4]:
print("Winter count:", (master_df["season"] == "Winter").sum())
print("\nFull time_of_day_bucket distribution:")
print(master_df["time_of_day_bucket"].value_counts().sort_values(ascending=False))
print("\nSum check — buckets should total to full dataset:", master_df["time_of_day_bucket"].value_counts().sum(), "vs", len(master_df))

Winter count: 116302

Full time_of_day_bucket distribution:
time_of_day_bucket
Midday          184238
Evening Rush    123623
Evening          91195
Morning Rush     85559
Night            29186
Name: count, dtype: int64

Sum check — buckets should total to full dataset: 513801 vs 513801


Task 14.3 — Drop Raw Date/Time Fields After Validation

With month, season, is_weekend, hour_of_day, and time_of_day_bucket all validated (sums match the full dataset, ranges are plausible), the raw date and time string columns are no longer needed as model inputs. date_parsed is retained temporarily for potential future temporal analysis (e.g., year-over-year trends) rather than dropped immediately.

In [5]:
master_df = master_df.drop(columns=["date", "time"])

print("Columns after drop:", master_df.shape[1])
print("\nConfirm derived time features present:")
print([col for col in master_df.columns if col in [
    "date_parsed", "day_of_week", "is_weekend", "month", "season", "hour_of_day", "time_of_day_bucket"
]])

Columns after drop: 65

Confirm derived time features present:
['day_of_week', 'date_parsed', 'hour_of_day', 'month', 'season', 'is_weekend', 'time_of_day_bucket']


Task 15 — Engineer/Select Geographic Features

Validate latitude/longitude ranges, assess the cardinality of local_authority_district and lsoa_of_collision_location, and decide whether each geographic field is retained as-is, grouped, or excluded — balancing spatial signal against encoding dimensionality and generalization risk, per Task 15's requirements.

In [8]:
# Latitude/longitude range validation (UK bounds roughly: lat 49-61, lon -8 to 2)
print("Latitude range:", master_df["latitude"].min(), "to", master_df["latitude"].max())
print("Longitude range:", master_df["longitude"].min(), "to", master_df["longitude"].max())
print("Missing latitude/longitude:", master_df["latitude"].isna().sum(), "/", master_df["longitude"].isna().sum())

# local_authority_district cardinality
print("\nlocal_authority_district unique categories:", master_df["local_authority_district"].nunique())
district_counts = master_df["local_authority_district"].value_counts()
print("Top 5 districts by count:")
print(district_counts.head())
print("Districts with fewer than 50 records (rare categories):", (district_counts < 50).sum())

# LSOA cardinality (corrected column name)
lsoa_col = "lsoa_of_accident_location"
print(f"\n{lsoa_col} unique categories:", master_df[lsoa_col].nunique())
print(f"Missing {lsoa_col}:", master_df[lsoa_col].isna().sum())
lsoa_counts = master_df[lsoa_col].value_counts()
print(f"LSOAs with only 1 record:", (lsoa_counts == 1).sum())

Latitude range: 49.91221 to 60.500008
Longitude range: -7.486852 to 1.759829
Missing latitude/longitude: 53 / 53

local_authority_district unique categories: 15
Top 5 districts by count:
local_authority_district
-1      513607
 241        32
 245        31
 243        27
 240        25
Name: count, dtype: int64
Districts with fewer than 50 records (rare categories): 14

lsoa_of_accident_location unique categories: 35630
Missing lsoa_of_accident_location: 0
LSOAs with only 1 record: 1840


In [7]:
lsoa_cols = [col for col in master_df.columns if "lsoa" in col.lower()]
print("Columns containing 'lsoa':", lsoa_cols)

Columns containing 'lsoa': ['lsoa_of_accident_location']


Task 15 — Final Geographic Feature Decision
local_authority_district: Excluded — 99.96% missing (sentinel -1), unusable.
latitude / longitude: Retained, but reserved for spatial/temporal hotspot analysis (heatmaps of collision density by location, day, and time bucket), not used as a standard scaled numeric feature in the classifier.
lsoa_of_accident_location: Retained for the same analysis purpose — provides named-area context for the hotspot maps (e.g., labeling which specific neighborhood a hotspot corresponds to) — but excluded from the classifier's direct inputs due to its 35,630-category cardinality.

These three fields are tagged analysis_only and combined with the engineered time features (day_of_week, hour_of_day, time_of_day_bucket) to support the risk-by-location-and-time analysis layer of the application, separate from the severity prediction model.

In [9]:
# Tag geographic + temporal fields reserved for hotspot/heatmap analysis (not model inputs)
analysis_only_cols = ["latitude", "longitude", "lsoa_of_accident_location"]

# Quick validation: confirm no missing values in the two fields we'll actually plot
print("Latitude missing:", master_df["latitude"].isna().sum())
print("Longitude missing:", master_df["longitude"].isna().sum())
print("Rows usable for hotspot mapping:", master_df[["latitude", "longitude"]].dropna().shape[0])

# Preview what a single hotspot slice would look like (e.g., Friday evening rush)
sample_slice = master_df[
    (master_df["day_of_week"] == 6) &  # Friday
    (master_df["time_of_day_bucket"] == "Evening Rush")
]
print("\nExample slice — Friday, Evening Rush collisions:", len(sample_slice))
print(sample_slice[["latitude", "longitude", "lsoa_of_accident_location", "collision_severity"]].head())

Latitude missing: 53
Longitude missing: 53
Rows usable for hotspot mapping: 513748

Example slice — Friday, Evening Rush collisions: 20842
     latitude  longitude lsoa_of_accident_location  collision_severity
11   53.82658   -3.01162                 E01012722                   3
35   52.93982   -1.18428                 E01013928                   3
70   51.41208   -0.96013                 E01016679                   3
104  55.23901   -1.75556                 E01027450                   3
105  55.70672   -1.99633                 E01027380                   3


Task 16 — Candidate Feature Inventory

Consolidate all fields into a single feature dictionary recording source table, data type, and current status (candidate model input / analysis-only / excluded), based on decisions made in Tasks 8-15. This is not yet the final leakage-reviewed feature set — Phase 6 will apply the prediction-point audit on top of this inventory.

In [10]:
feature_inventory = []

# Targets
feature_inventory.append({"feature": "collision_severity", "source": "Collision", "type": "categorical", "status": "TARGET (classification)"})
feature_inventory.append({"feature": "number_of_casualties", "source": "Collision", "type": "numeric", "status": "TARGET (regression)"})

# Identifiers / reference
feature_inventory.append({"feature": "collision_index", "source": "Collision", "type": "identifier", "status": "reference only"})

# Collision context — candidates
for col in ["light_conditions", "weather_conditions", "road_surface_conditions", "road_type",
            "speed_limit", "junction_detail", "junction_control", "urban_or_rural_area",
            "special_conditions_at_site", "carriageway_hazards", "trunk_road_flag",
            "first_road_class", "number_of_vehicles"]:
    feature_inventory.append({"feature": col, "source": "Collision", "type": "collision context", "status": "candidate"})

# Geographic — analysis-only
for col in ["latitude", "longitude", "lsoa_of_accident_location"]:
    feature_inventory.append({"feature": col, "source": "Collision", "type": "geographic", "status": "analysis-only (hotspot mapping)"})

feature_inventory.append({"feature": "local_authority_district", "source": "Collision", "type": "geographic", "status": "excluded (99.96% missing)"})

# Engineered time features
for col in ["day_of_week", "is_weekend", "month", "season", "hour_of_day", "time_of_day_bucket"]:
    feature_inventory.append({"feature": col, "source": "engineered", "type": "temporal", "status": "candidate"})

# Vehicle-derived candidates
for col in ["has_car", "has_motorcycle", "has_pedal_cycle", "has_hgv", "has_bus_or_coach",
            "has_van_or_goods_light", "has_other", "has_undocumented_code_33",
            "vehicle_type_diversity", "min_driver_age", "max_driver_age", "avg_driver_age"]:
    feature_inventory.append({"feature": col, "source": "Vehicle (aggregated)", "type": "vehicle-derived", "status": "conditional candidate — prediction-point review pending"})

# Casualty-derived — analysis-only (post-collision outcomes)
for col in ["pedestrian_involved", "has_passenger_casualty", "has_child_casualty",
            "has_elderly_casualty", "worst_casualty_severity"]:
    feature_inventory.append({"feature": col, "source": "Casualty (aggregated)", "type": "casualty outcome", "status": "analysis-only (post-collision, excluded from prospective model)"})

feature_df = pd.DataFrame(feature_inventory)

print("Total fields in inventory:", len(feature_df))
print("\nStatus breakdown:")
print(feature_df["status"].value_counts())

feature_df

Total fields in inventory: 43

Status breakdown:
status
candidate                                                          19
conditional candidate — prediction-point review pending            12
analysis-only (post-collision, excluded from prospective model)     5
analysis-only (hotspot mapping)                                     3
TARGET (classification)                                             1
TARGET (regression)                                                 1
reference only                                                      1
excluded (99.96% missing)                                           1
Name: count, dtype: int64


,feature,source,type,status
0,collision_severity,Collision,categorical,TARGET (classification)
1,number_of_casualties,Collision,numeric,TARGET (regression)
2,collision_index,Collision,identifier,reference only
3,light_conditions,Collision,collision context,candidate
4,weather_conditions,Collision,collision context,candidate
5,road_surface_conditions,Collision,collision context,candidate
6,road_type,Collision,collision context,candidate
7,speed_limit,Collision,collision context,candidate
8,junction_detail,Collision,collision context,candidate
9,junction_control,Collision,collision context,candidate


In [11]:
# Add missingness note to the geographic analysis-only rows
missing_note = f"53 records ({53/len(master_df)*100:.3f}%) missing lat/long — excluded from heatmap, retained in dataset"

feature_df.loc[feature_df["feature"].isin(["latitude", "longitude"]), "note"] = missing_note
feature_df.loc[feature_df["feature"] == "lsoa_of_accident_location", "note"] = "no missing values"

feature_df[feature_df["type"] == "geographic"]

,feature,source,type,status,note
16,latitude,Collision,geographic,analysis-only (hotspot mapping),53 records (0.010%) missing lat/long — exclude...
17,longitude,Collision,geographic,analysis-only (hotspot mapping),53 records (0.010%) missing lat/long — exclude...
18,lsoa_of_accident_location,Collision,geographic,analysis-only (hotspot mapping),no missing values
19,local_authority_district,Collision,geographic,excluded (99.96% missing),NaN


Save Phase 5 Outputs

Save the feature-engineered Master Dataset (with time features added, raw date/time dropped) and the feature inventory table, so Phase 6 (Prediction-Point Review) can load them independently.

In [12]:
output_path = Path("C:/Users/ACER/Downloads/New folder/cleaned_datasets")

master_df.to_csv(output_path / "master_dataset_v2_engineered.csv", index=False)
feature_df.to_csv(output_path / "feature_inventory.csv", index=False)

print("Saved:")
print("Master Dataset (v2, engineered):", master_df.shape)
print("Feature inventory:", feature_df.shape)

Saved:
Master Dataset (v2, engineered): (513801, 65)
Feature inventory: (43, 5)


NameError: name 'master_df' is not defined